# 80 Region Talk migration reconciliation

Pure contract worker for row-kind accounting; it never reads YDB or writes PostgreSQL.

This notebook is a **typed producer**, not a database client. It reads one exact input manifest and writes one immutable result envelope. It must never mutate canonical PostgreSQL, YDB, a shared SQLite file or a publication target.

Adapter-pending stages fail closed per item until donor code, model revisions and shadow equivalence have been recorded.

In [ ]:
from __future__ import annotations

import json
import os
import tempfile
from pathlib import Path

from my_data_hub.notebooks.runtime import (
    NotebookResultBuilder,
    manifest_path_from_env,
)

STAGE_CONTRACTS = {'migration_reconciliation': 'region-talk.migration-reconciliation.v1'}
MODEL = {'provider': 'none', 'name': 'migration-reconciliation', 'version': 'v1', 'task': 'migration-audit'}

In [ ]:
builder = NotebookResultBuilder(
    manifest_path=manifest_path_from_env(),
    code_revision=os.environ.get("MY_DATA_HUB_CODE_REVISION", "UNPINNED"),
    runtime_name=os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "local-notebook"),
)
manifest = builder.manifest
expected_contract = STAGE_CONTRACTS.get(manifest.stage)
if expected_contract is None:
    raise RuntimeError(f"unsupported stage for this notebook: {manifest.stage}")
if manifest.stage_contract_version != expected_contract:
    raise RuntimeError(
        f"stage contract mismatch: {manifest.stage_contract_version} != {expected_contract}"
    )
print({"run_id": str(manifest.run_id), "stage": manifest.stage, "items": len(manifest.work_items)})

## Stage adapter

Replace only the adapter implementation after focused tests. Keep manifest validation, per-item accounting, fingerprints and result emission unchanged.

In [ ]:
from my_data_hub.workloads.region_talk.migration import (
    build_reconciliation_accounting,
)


def process_item(work_item: dict) -> dict:
    payload = work_item.get("payload", {})
    expected = payload.get("expected_by_kind")
    actual = payload.get("actual_rows")
    if not isinstance(expected, dict) or not isinstance(actual, list):
        raise ValueError("payload requires expected_by_kind object and actual_rows list")
    pairs = []
    for item in actual:
        if not isinstance(item, dict) or "row_kind" not in item:
            raise ValueError("each actual_rows item requires row_kind")
        pairs.append((str(item["row_kind"]), item.get("disposition")))
    return {
        "accounting": build_reconciliation_accounting(
            expected_by_kind=expected,
            actual_rows=pairs,
        )
    }

In [ ]:
for item in manifest.work_items:
    work_item = item.model_dump(mode="json")
    try:
        result = process_item(work_item)
    except NotImplementedError as exc:
        builder.add_failure(
            work_item_id=item.work_item_id,
            code="PROCESSOR_ADAPTER_NOT_PORTED",
            message=str(exc),
            retryable=False,
        )
    except Exception as exc:
        builder.add_failure(
            work_item_id=item.work_item_id,
            code="PROCESSOR_FAILURE",
            message=str(exc),
            retryable=True,
            details={"exception_type": type(exc).__name__},
        )
    else:
        builder.add_success(
            work_item_id=item.work_item_id,
            input_fingerprint=item.input_fingerprint,
            result=result,
        )

In [ ]:
result = builder.build(MODEL)
output_path = Path(
    os.environ.get(
        "MY_DATA_HUB_NOTEBOOK_RESULT_PATH",
        "/kaggle/working/result.json",
    )
)
output_path.parent.mkdir(parents=True, exist_ok=True)
payload = json.dumps(
    result, ensure_ascii=False, sort_keys=True, separators=(",", ":")
).encode("utf-8")
descriptor, temporary_name = tempfile.mkstemp(
    prefix=".result.", suffix=".tmp", dir=output_path.parent
)
try:
    with os.fdopen(descriptor, "wb") as handle:
        handle.write(payload)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary_name, output_path)
finally:
    if os.path.exists(temporary_name):
        os.unlink(temporary_name)
print({"status": result["status"], "result": str(output_path), "failures": len(result["failures"])})